# Phase 6 — Feature Engineering

In the previous phases, we explored and analyzed the customer churn dataset to understand its structure, quality, and relationships between variables.

The next step is Feature Engineering, where we transform the analyzed data into a format suitable for machine learning algorithms.

The objective of feature engineering is to improve the quality of the input data by selecting relevant features, transforming variables, encoding categorical data, and scaling numerical features where necessary.

Unlike Exploratory Data Analysis (EDA), which focuses on understanding the data, feature engineering focuses on preparing the data for predictive modeling.

By the end of this phase, we will have:

- Selected the most relevant features.
- Encoded categorical variables.
- Scaled numerical features where appropriate.
- Created additional features if they improve predictive performance.
- Generated the final feature matrix (`X`) and target vector (`y`) ready for machine learning.

## Part A — Feature Selection

Feature Selection is the process of identifying the most relevant features for building a machine learning model.

Not every feature in a dataset contributes equally to prediction. Some features may carry useful information, while others may be redundant, irrelevant, or introduce unnecessary noise into the model.

The objective of feature selection is to retain only those features that improve the model's ability to predict customer churn while excluding features that do not provide meaningful predictive value.

The decisions made during this step are based on:

- Insights obtained during Exploratory Data Analysis (EDA).
- Statistical relationships between features and the target variable.
- Domain knowledge of the customer churn problem.
- Practical considerations for building an efficient and interpretable model.

For this project, we will review every feature before deciding whether it should be retained, transformed, or removed prior to model training.

### Step 1: Remove Identifier Columns

The first preprocessing step is to remove identifier columns that do not contribute to predicting customer churn.

Identifier columns contain unique values for each record and do not provide meaningful patterns for machine learning models. Including such features may increase model complexity without improving predictive performance.

Based on the insights from Exploratory Data Analysis (EDA), the `customerID` column serves only as a unique identifier and does not contain predictive information.

Therefore, it will be removed before further preprocessing.

**Columns to Remove:**

- `customerID`

In [78]:
import pandas as pd

# df = pd.read_csv("../data/processed/customer_churn_clean.csv")
# df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv("../data/processed/v1_customer_churn_processed.csv")

In [79]:
# Helper Function 1

def remove_identifier_columns(df, columns):
    """
    Remove identifier columns from the dataset.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.

    columns : list
        List of identifier columns to remove.

    Returns
    -------
    pandas.DataFrame
        Dataset after removing identifier columns.
    """

    df = df.drop(columns=columns, errors="ignore")

    print("Removed Identifier Columns:")
    print(columns)

    print(f"\nUpdated Dataset Shape: {df.shape}")

    return df

In [80]:
identifier_columns = ["customerID"]

df = remove_identifier_columns(
    df,
    identifier_columns
)

Removed Identifier Columns:
['customerID']

Updated Dataset Shape: (7032, 20)


### Step 2: Separate Features and Target

Machine learning models learn patterns by using input features to predict a target variable.

Therefore, before applying any preprocessing techniques, it is important to separate the dataset into:

- **Feature Matrix (`X`)**: Contains all input variables used for prediction.
- **Target Vector (`y`)**: Contains the variable that the model will learn to predict.

For this project:

- **Input Features (`X`)** include all customer-related attributes after removing the identifier column.
- **Target Variable (`y`)** is the `Churn` column.

Separating the features and target at this stage ensures that future preprocessing steps, such as encoding and scaling, are applied only to the input features while leaving the target variable unchanged.

In [81]:
# Helper Function 2

def separate_features_target(df, target_column):
    """
    Separate the dataset into feature matrix (X)
    and target vector (y).

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.

    target_column : str
        Name of the target variable.

    Returns
    -------
    X : pandas.DataFrame
        Feature matrix.

    y : pandas.Series
        Target variable.
    """

    if target_column not in df.columns:
        raise ValueError(f"'{target_column}' not found in the dataset.")

    X = df.drop(columns=target_column)
    y = df[target_column]

    print("Features Shape :", X.shape)
    print("Target Shape   :", y.shape)

    return X, y

In [82]:
X, y = separate_features_target(
    df,
    target_column="Churn"
)

Features Shape : (7032, 19)
Target Shape   : (7032,)


In [83]:
print("Feature Columns:")
display(X.head())

print("Target Variable:")
display(y.head())

Feature Columns:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


Target Variable:


0     No
1     No
2    Yes
3     No
4    Yes
Name: Churn, dtype: object

## Part B — Feature Transformation

Before encoding categorical variables or scaling numerical features, it is important to determine whether any features require transformation.

Feature transformation involves modifying the representation of existing variables to make them more suitable for machine learning algorithms.

Common transformations include:

- Converting incorrect data types.
- Handling skewed numerical distributions.
- Applying mathematical transformations when appropriate.

However, not every dataset requires extensive transformations.

For this project, the transformations applied will be guided by the findings from Exploratory Data Analysis (EDA) rather than applying preprocessing techniques indiscriminately.

The objective is to perform only those transformations that improve the quality and usability of the data while preserving interpretability.

In [84]:
X.dtypes

gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
dtype: object

In [85]:
X[["tenure", "MonthlyCharges", "TotalCharges"]].skew()

tenure            0.237731
MonthlyCharges   -0.222103
TotalCharges      0.961642
dtype: float64

### Step 1: Analyze Numerical Feature Distribution

Before applying any mathematical transformations, it is important to evaluate the distribution of the numerical features.

Many machine learning algorithms do not require normally distributed features. However, highly skewed distributions can sometimes affect model performance, particularly for algorithms that assume approximately symmetric feature distributions.

The objective of this step is to determine whether any numerical features require additional transformations, such as logarithmic or power transformations.

For this project, we will examine the skewness of the numerical features and apply transformations only if they provide a meaningful benefit.

In [86]:
# Helper Function 1

def analyze_numerical_skewness(df, numerical_columns):
    """
    Analyze the skewness of numerical features.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.

    numerical_columns : list
        List of numerical feature names.

    Returns
    -------
    pandas.DataFrame
        Skewness summary.
    """

    skewness = (
        df[numerical_columns]
        .skew()
        .sort_values(key=abs, ascending=False)
        .to_frame(name="Skewness")
    )

    skewness["Interpretation"] = skewness["Skewness"].apply(
        lambda x: (
            "Highly Skewed" if abs(x) > 1
            else "Moderately Skewed" if abs(x) > 0.5
            else "Approximately Symmetric"
        )
    )

    display(skewness)

    return skewness

In [87]:
numerical_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

skewness_summary = analyze_numerical_skewness(
    df,
    numerical_columns
)

,Skewness,Interpretation
TotalCharges,0.961642,Moderately Skewed
tenure,0.237731,Approximately Symmetric
MonthlyCharges,-0.222103,Approximately Symmetric


### Observation

The skewness analysis indicates that:

- `tenure` is approximately symmetric.
- `MonthlyCharges` is approximately symmetric.
- `TotalCharges` exhibits moderate positive skewness.

Although `TotalCharges` is moderately skewed, the skewness is expected because customers with longer tenures naturally accumulate higher total charges over time.

Since the skewness is not severe and preserves meaningful business information, no mathematical transformation (such as logarithmic or power transformation) is applied.

Therefore, the numerical features will be retained in their original form, and the next preprocessing step will focus on encoding categorical variables.

## Part C — Encoding Categorical Variables

Most machine learning algorithms require numerical input features and cannot directly process categorical variables represented as text.

Therefore, categorical features must be converted into numerical representations while preserving the information they contain.

Several encoding techniques exist, including:

- Label Encoding
- One-Hot Encoding
- Ordinal Encoding
- Target Encoding

The appropriate encoding technique depends on the nature of the categorical feature.

For this customer churn dataset, the majority of categorical variables are **nominal**, meaning their categories have no natural ordering. Applying Label Encoding to such features may introduce an artificial ordinal relationship that does not exist.

Therefore, **One-Hot Encoding** is the most appropriate encoding technique for this dataset.

In this section, we will:

- Identify categorical features.
- Separate nominal and binary categorical variables.
- Apply the appropriate encoding strategy.
- Generate a machine learning-ready feature matrix.

### Step 1: Identify Categorical Features for Encoding

Before applying any encoding technique, it is important to identify the categorical features present in the dataset and determine the most appropriate encoding strategy for each.

For this project, categorical features are divided into two groups:

- **Binary categorical features**, which contain only two unique categories. These features will be encoded using Binary Encoding (0 and 1).
- **Multi-class categorical features**, which contain more than two categories. These features will be encoded using One-Hot Encoding.

Separating categorical features in this manner helps reduce unnecessary dimensions while preserving all relevant information for machine learning models.

In [88]:
# Helper Function 1

def identify_categorical_features(df):
    """
    Identify binary and multi-class categorical features.

    Parameters
    ----------
    df : pandas.DataFrame
        Feature matrix.

    Returns
    -------
    tuple
        (binary_features, multiclass_features)
    """

    categorical_features = df.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    binary_features = []
    multiclass_features = []

    for feature in categorical_features:

        unique_values = df[feature].dropna().unique()

        if len(unique_values) == 2:
            binary_features.append(feature)
        else:
            multiclass_features.append(feature)

    print("=" * 60)
    print("CATEGORICAL FEATURE SUMMARY")
    print("=" * 60)

    print("\nBinary Features")
    print(binary_features)

    print("\nMulti-class Features")
    print(multiclass_features)

    return binary_features, multiclass_features

In [89]:
binary_features, multiclass_features = identify_categorical_features(X)

CATEGORICAL FEATURE SUMMARY

Binary Features
['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

Multi-class Features
['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']


### Step 2: Encode Binary Categorical Features

Binary categorical features contain only two unique categories. Instead of applying One-Hot Encoding, these features can be efficiently represented using a single binary value.

For this dataset, binary features will be encoded as:

- The first category → 0
- The second category → 1

This approach reduces the number of generated features while preserving all information contained in the original variables.

The mapping for each feature will be displayed to ensure transparency and reproducibility.

In [90]:
# Helper Function 2

def encode_binary_features(df, binary_features):
    """
    Encode binary categorical features using 0 and 1.

    Parameters
    ----------
    df : pandas.DataFrame
        Feature matrix.

    binary_features : list
        List of binary categorical features.

    Returns
    -------
    pandas.DataFrame
        DataFrame with encoded binary features.
    """

    df = df.copy()

    mapping_summary = {}

    for feature in binary_features:

        categories = sorted(df[feature].dropna().unique())

        mapping = {
            categories[0]: 0,
            categories[1]: 1
        }

        df[feature] = df[feature].map(mapping)

        mapping_summary[feature] = mapping

    print("=" * 60)
    print("BINARY ENCODING SUMMARY")
    print("=" * 60)

    for feature, mapping in mapping_summary.items():
        print(f"{feature}: {mapping}")

    return df

In [91]:
X = encode_binary_features(
    X,
    binary_features
)

BINARY ENCODING SUMMARY
gender: {'Female': 0, 'Male': 1}
Partner: {'No': 0, 'Yes': 1}
Dependents: {'No': 0, 'Yes': 1}
PhoneService: {'No': 0, 'Yes': 1}
PaperlessBilling: {'No': 0, 'Yes': 1}


### Step 3: Encode Multi-Class Categorical Features

Unlike binary categorical features, multi-class categorical features contain more than two unique categories.

Applying Label Encoding to these features would introduce an artificial ordinal relationship between categories, which could negatively affect model performance.

Therefore, One-Hot Encoding is used to transform each category into a separate binary feature.

To avoid multicollinearity, the first category of each feature is dropped using `drop_first=True`. This reduces redundancy while preserving the information required by machine learning models.

In [92]:
# Helper Function 3

def one_hot_encode_features(df, multiclass_features):
    """
    Apply One-Hot Encoding to multi-class categorical features.

    Parameters
    ----------
    df : pandas.DataFrame
        Feature matrix.

    multiclass_features : list
        List of multi-class categorical features.

    Returns
    -------
    pandas.DataFrame
        DataFrame after One-Hot Encoding.
    """

    original_columns = df.shape[1]

    df_encoded = pd.get_dummies(
        df,
        columns=multiclass_features,
        drop_first=True,
        dtype=int
    )

    print("=" * 60)
    print("ONE-HOT ENCODING SUMMARY")
    print("=" * 60)
    print(f"Original Features : {original_columns}")
    print(f"Encoded Features  : {df_encoded.shape[1]}")
    print(f"New Features Added: {df_encoded.shape[1] - original_columns}")

    return df_encoded

In [93]:
X = one_hot_encode_features(
    X,
    multiclass_features
)

ONE-HOT ENCODING SUMMARY
Original Features : 19
Encoded Features  : 30
New Features Added: 11


### Step 4: Validate the Encoded Dataset

After encoding the categorical features, it is important to verify that the dataset is suitable for machine learning.

The validation includes checking:

- The shape of the encoded dataset.
- The data types of all features.
- Whether any categorical (`object`) columns remain.
- Whether any missing values were introduced during encoding.

This final validation ensures that the feature matrix is fully numeric and ready for subsequent preprocessing steps, such as feature scaling and model training.

In [94]:
# Helper Function 4

def validate_encoded_dataset(df):
    """
    Validate the encoded dataset before moving to the next
    preprocessing stage.

    Parameters
    ----------
    df : pandas.DataFrame
        Encoded feature matrix.

    Returns
    -------
    None
    """

    print("=" * 60)
    print("ENCODED DATASET VALIDATION")
    print("=" * 60)

    print(f"Dataset Shape      : {df.shape}")

    object_columns = df.select_dtypes(include=["object"]).columns.tolist()

    print(f"Object Columns     : {len(object_columns)}")

    if object_columns:
        print(object_columns)
    else:
        print("✓ No object columns remain.")

    missing_values = df.isnull().sum().sum()

    print(f"Missing Values     : {missing_values}")

    if missing_values == 0:
        print("✓ No missing values detected.")
    else:
        print("⚠ Missing values exist.")

    print("\nData Types Summary")
    print(df.dtypes.value_counts())

In [95]:
validate_encoded_dataset(X)

ENCODED DATASET VALIDATION
Dataset Shape      : (7032, 30)
Object Columns     : 0
✓ No object columns remain.
Missing Values     : 0
✓ No missing values detected.

Data Types Summary
int64      28
float64     2
Name: count, dtype: int64


In [96]:
# Let's investigate where they are

# Instead of guessing, let's identify the affected columns.

def inspect_missing_values(df):
    """
    Display columns containing missing values.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.

    Returns
    -------
    pandas.DataFrame
        Summary of missing values.
    """

    missing_summary = (
        df.isnull()
          .sum()
          .loc[lambda x: x > 0]
          .sort_values(ascending=False)
          .to_frame(name="Missing Values")
    )

    if missing_summary.empty:
        print("✓ No missing values found.")
    else:
        display(missing_summary)

    return missing_summary


In [97]:
missing_summary = inspect_missing_values(X)

✓ No missing values found.


## Part D — Feature Scaling

Machine learning algorithms often perform better when numerical features are on a similar scale.

Features with larger numerical ranges can dominate distance calculations or gradient updates, potentially affecting model performance.

However, not all machine learning algorithms require feature scaling. Tree-based algorithms such as Decision Trees and Random Forests are generally unaffected by differences in feature scales, whereas algorithms like Logistic Regression, Support Vector Machines (SVM), K-Nearest Neighbors (KNN), and Neural Networks benefit significantly from scaled features.

In this project, only the continuous numerical features will be scaled. Binary and one-hot encoded features already contain values of 0 and 1, making additional scaling unnecessary.

The scaling process will therefore include:

- Identifying the continuous numerical features.
- Applying an appropriate scaling technique.
- Verifying that the transformed features have been scaled correctly.

### Step 1: Identify Continuous Numerical Features

The dataset now contains different types of numerical features:

- Continuous numerical features
- Binary encoded features
- One-Hot encoded features

Since binary and one-hot encoded features already lie within a common range (0 and 1), only the continuous numerical features require scaling.

In [98]:
# Helper Function 1

def identify_features_for_scaling(df):
    """
    Identify continuous numerical features that require scaling.

    Parameters
    ----------
    df : pandas.DataFrame
        Encoded feature matrix.

    Returns
    -------
    list
        Continuous numerical feature names.
    """

    continuous_features = [
        "tenure",
        "MonthlyCharges",
        "TotalCharges"
    ]

    print("=" * 60)
    print("FEATURES SELECTED FOR SCALING")
    print("=" * 60)

    for feature in continuous_features:
        print(f"• {feature}")

    return continuous_features

In [99]:
continuous_features = identify_features_for_scaling(X)

FEATURES SELECTED FOR SCALING
• tenure
• MonthlyCharges
• TotalCharges


### Step 2: Apply Standard Scaling

Several scaling techniques are commonly used in machine learning, including:

- Min-Max Scaling
- Standard Scaling
- Robust Scaling

For this project, **StandardScaler** is selected because it standardizes continuous numerical features by centering them around a mean of 0 and scaling them to a standard deviation of 1.

This scaling technique is widely used and performs well for many machine learning algorithms, including Logistic Regression, Support Vector Machines, and Neural Networks.

Only the continuous numerical features will be transformed, while binary and one-hot encoded features will remain unchanged.

In [100]:
# Helper Function 2

from sklearn.preprocessing import StandardScaler


def scale_continuous_features(df, continuous_features):
    """
    Scale continuous numerical features using StandardScaler.

    Parameters
    ----------
    df : pandas.DataFrame
        Encoded feature matrix.

    continuous_features : list
        Features to scale.

    Returns
    -------
    tuple
        (scaled_dataframe, fitted_scaler)
    """

    df = df.copy()

    scaler = StandardScaler()

    df[continuous_features] = scaler.fit_transform(
        df[continuous_features]
    )

    print("=" * 60)
    print("FEATURE SCALING SUMMARY")
    print("=" * 60)

    print(f"Scaling Method : {scaler.__class__.__name__}")
    print(f"Features Scaled: {len(continuous_features)}")

    return df, scaler

In [101]:
X, scaler = scale_continuous_features(
    X,
    continuous_features
)

FEATURE SCALING SUMMARY
Scaling Method : StandardScaler
Features Scaled: 3


### Step 3: Validate Feature Scaling

After applying feature scaling, it is important to verify that the transformation has been performed correctly.

For features scaled using **StandardScaler**, the expected characteristics are:

- Mean approximately equal to **0**
- Standard deviation approximately equal to **1**

Validating the transformed features ensures that the scaling process has been successfully applied and that the dataset is ready for model training.

In [102]:
# Helper Function 3

def validate_feature_scaling(df, continuous_features):
    """
    Validate the scaled numerical features.

    Parameters
    ----------
    df : pandas.DataFrame
        Feature matrix after scaling.

    continuous_features : list
        Continuous numerical features.

    Returns
    -------
    pandas.DataFrame
        Scaling summary.
    """

    scaling_summary = df[continuous_features].agg(
        ["mean", "std", "min", "max"]
    ).T

    scaling_summary = scaling_summary.rename(
        columns={
            "mean": "Mean",
            "std": "Std Dev",
            "min": "Minimum",
            "max": "Maximum"
        }
    )

    print("=" * 60)
    print("FEATURE SCALING VALIDATION")
    print("=" * 60)

    display(scaling_summary.round(4))

    return scaling_summary

In [103]:
scaling_summary = validate_feature_scaling(
    X,
    continuous_features
)

FEATURE SCALING VALIDATION


,Mean,Std Dev,Minimum,Maximum
tenure,-0.0,1.0001,-1.2802,1.6126
MonthlyCharges,0.0,1.0001,-1.5473,1.7934
TotalCharges,-0.0,1.0001,-0.9991,2.8243


## Part E — Feature Creation

Feature engineering involves creating new features from existing data to improve the predictive capability of machine learning models.

Rather than generating arbitrary features, only features with a clear business interpretation and potential predictive value should be created.

For this customer churn prediction project, a new feature named **TotalServices** will be engineered.

This feature represents the total number of optional services subscribed to by each customer.

The underlying assumption is that customers who subscribe to a greater number of services are generally more engaged with the company and may exhibit different churn behavior compared to customers who use only a few services.

The engineered feature will later be evaluated during feature selection to determine whether it contributes positively to model performance.

### Step 1: Create TotalServices Feature

The **TotalServices** feature counts the number of subscribed services for each customer.

Each subscribed service contributes a value of **1**, while non-subscribed services contribute **0**.

This feature provides a compact representation of customer engagement and service adoption.

In [104]:
def create_total_services_feature(df):
    """
    Create the TotalServices feature by counting the
    number of subscribed services.

    Parameters
    ----------
    df : pandas.DataFrame
        Encoded feature matrix.

    Returns
    -------
    pandas.DataFrame
        Dataset with TotalServices feature.
    """

    df = df.copy()

    service_columns = [
        "PhoneService",
        "MultipleLines_Yes",
        "InternetService_Fiber optic",
        "InternetService_No",
        "OnlineSecurity_Yes",
        "OnlineBackup_Yes",
        "DeviceProtection_Yes",
        "TechSupport_Yes",
        "StreamingTV_Yes",
        "StreamingMovies_Yes"
    ]

    df["TotalServices"] = (
        df[service_columns]
        .sum(axis=1)
    )

    print("=" * 60)
    print("FEATURE CREATED")
    print("=" * 60)
    print("New Feature : TotalServices")
    print(f"Range       : {df['TotalServices'].min()} - {df['TotalServices'].max()}")

    return df

In [105]:
X = create_total_services_feature(X)

FEATURE CREATED
New Feature : TotalServices
Range       : 0 - 9


### Step 2: Validate the Engineered Feature

After creating the **TotalServices** feature, it is important to verify that it has been generated correctly.

The validation includes:

- Confirming that the feature exists.
- Reviewing its descriptive statistics.
- Checking its distribution.
- Ensuring that the values fall within the expected range.

In [106]:
def validate_engineered_features(df):
    """
    Validate engineered features.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset containing engineered features.

    Returns
    -------
    None
    """

    print("=" * 60)
    print("ENGINEERED FEATURE VALIDATION")
    print("=" * 60)

    print("\nSummary Statistics\n")

    display(
        df[["TotalServices"]]
        .describe()
        .T
        .round(2)
    )

    print("\nValue Counts\n")

    display(
        df["TotalServices"]
        .value_counts()
        .sort_index()
        .to_frame(name="Count")
    )

In [107]:
validate_engineered_features(X)

ENGINEERED FEATURE VALIDATION

Summary Statistics



,count,mean,std,min,25%,50%,75%,max
TotalServices,7032.0,4.02,2.07,0.0,2.0,4.0,6.0,9.0



Value Counts



,Count
TotalServices,
0,80
1,284
2,1828
3,1218
4,883
5,897
6,809
7,583
8,329


## Part F — Remove Redundant Features

Feature engineering can introduce new variables that overlap with information already present in the dataset.

Before preparing the final dataset, it is important to identify redundant features that may increase model complexity without providing additional predictive value.

In this section, we will:

- Review the engineered feature alongside related features.
- Evaluate potential redundancy.
- Decide whether all engineered and original features should be retained for model training.

The goal is to produce a feature set that is informative, interpretable, and suitable for machine learning models.

In [109]:
# Step 1 — Correlation Analysis
# We'll compare TotalServices with the other numerical features.

def analyze_engineered_feature_correlation(df):
    """
    Analyze the correlation of the engineered feature
    with other numerical features.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset containing engineered features.

    Returns
    -------
    pandas.DataFrame
        Correlation values.
    """

    numerical_columns = [
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "TotalServices"
    ]

    correlation = (
        df[numerical_columns]
        .corr()["TotalServices"]
        .sort_values(ascending=False)
        .to_frame(name="Correlation")
    )

    print("=" * 60)
    print("ENGINEERED FEATURE CORRELATION")
    print("=" * 60)

    display(correlation.round(3))

    return correlation

In [110]:
correlation = analyze_engineered_feature_correlation(X)

ENGINEERED FEATURE CORRELATION


,Correlation
TotalServices,1.000
MonthlyCharges,0.838
TotalCharges,0.807
tenure,0.520


### Observation

The engineered feature `TotalServices` exhibits a strong positive correlation with `MonthlyCharges` and `TotalCharges`. This relationship is expected, as customers subscribing to more services generally incur higher monthly and cumulative charges.

Despite this correlation, `TotalServices` captures a different aspect of customer behavior—overall service adoption—while `MonthlyCharges` and `TotalCharges` represent financial expenditure.

Furthermore, the individual service features provide detailed information about the specific services a customer has subscribed to, which cannot be reconstructed from the aggregated `TotalServices` feature alone.

Based on these observations, no features are removed. Both the engineered feature and the original service-related features are retained for model training.

## Part G — Final Dataset Preparation

The final step in the feature engineering pipeline is to verify that the dataset is fully prepared for machine learning.

At this stage, the dataset should satisfy the following conditions:

- All features are numerical.
- No missing values are present.
- All required feature engineering steps have been completed.
- The feature matrix (`X`) and target variable (`y`) are ready for model training.

Finally, the engineered dataset will be saved so it can be reused during model development without repeating the preprocessing steps.

### Step 1: Validate the Final Dataset

Before proceeding to model training, a final validation is performed to ensure the dataset is complete, consistent, and suitable for machine learning algorithms.

The validation checks include:

- Dataset dimensions.
- Missing values.
- Duplicate records.
- Data types.
- Target variable distribution.

In [111]:
def validate_final_dataset(X, y):
    """
    Perform a final validation of the feature matrix
    and target variable.

    Parameters
    ----------
    X : pandas.DataFrame
        Final feature matrix.

    y : pandas.Series
        Target variable.

    Returns
    -------
    None
    """

    print("=" * 60)
    print("FINAL DATASET VALIDATION")
    print("=" * 60)

    print(f"Feature Matrix Shape : {X.shape}")
    print(f"Target Shape         : {y.shape}")

    print(f"\nMissing Values (X)   : {X.isnull().sum().sum()}")
    print(f"Missing Values (y)   : {y.isnull().sum()}")

    print(f"\nDuplicate Rows (X)   : {X.duplicated().sum()}")

    object_columns = X.select_dtypes(include="object").columns.tolist()

    print(f"Object Columns       : {len(object_columns)}")

    if len(object_columns) == 0:
        print("✓ All features are numeric.")

    print("\nTarget Distribution")

    display(
        y.value_counts().to_frame("Count")
    )

In [112]:
validate_final_dataset(X, y)

FINAL DATASET VALIDATION
Feature Matrix Shape : (7032, 31)
Target Shape         : (7032,)

Missing Values (X)   : 0
Missing Values (y)   : 0

Duplicate Rows (X)   : 40
Object Columns       : 0
✓ All features are numeric.

Target Distribution


,Count
Churn,
No,5163
Yes,1869


In [114]:
pd.concat([X, y], axis=1).duplicated().sum()

np.int64(22)

In [117]:
from pathlib import Path
import pandas as pd


def save_feature_engineered_dataset(X, y, file_name):
    """
    Save the final feature-engineered dataset.

    Parameters
    ----------
    X : pandas.DataFrame
        Final feature matrix.

    y : pandas.Series
        Target variable.

    file_name : str
        Name of the output CSV file.

    Returns
    -------
    pandas.DataFrame
        Final saved dataset.
    """

    final_dataset = X.copy()
    final_dataset["Churn"] = y.values

    output_path = Path("../data/processed")
    output_path.mkdir(parents=True, exist_ok=True)

    file_path = output_path / file_name

    final_dataset.to_csv(file_path, index=False)

    print("=" * 60)
    print("FEATURE ENGINEERED DATASET SAVED")
    print("=" * 60)
    print(f"Location : {file_path}")
    print(f"Shape    : {final_dataset.shape}")

    return final_dataset

In [118]:
final_dataset = save_feature_engineered_dataset(
    X,
    y,
    "v1_feature_engineered_customer_churn.csv"
)

FEATURE ENGINEERED DATASET SAVED
Location : ..\data\processed\v1_feature_engineered_customer_churn.csv
Shape    : (7032, 32)
